## Semantic Analysis-Based Sockpuppet Detection in Wikipedia

This section explores sockpuppet detection through semantic analysis, focusing on understanding and leveraging the meanings and relationships of words within the Wikipedia dataset. The method employs various NLP techniques to analyze the text data, aiming to identify patterns or anomalies indicative of sockpuppet behavior based solely on content semantics.


In [1]:
# Importing necessary libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load the pre-split training and testing sets
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Combine the training and testing text to fit the vectorizer
all_text = pd.concat([train_data['edit_text'], test_data['edit_text']])

# Semantic Analysis using TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(all_text)  # Learn vocabulary and idf from training set
X_train = tfidf.transform(train_data['edit_text']).toarray()
X_test = tfidf.transform(test_data['edit_text']).toarray()

# Labels for the dataset
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.64      0.70      5382
           1       0.70      0.81      0.75      5382

    accuracy                           0.73     10764
   macro avg       0.74      0.73      0.73     10764
weighted avg       0.74      0.73      0.73     10764

Accuracy: 0.7289111854329245


## Sentiment Analysis-Based Sockpuppet Detection in Wikipedia

In this part of the notebook, sentiment analysis is used to detect sockpuppets on Wikipedia. The approach analyzes the emotional tone and subjective expressions in user contributions, looking for sentiment patterns that are often associated with deceptive or manipulative online behaviors characteristic of sockpuppets.


In [2]:
# Importing necessary libraries
import pandas as pd
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load the pre-split training and testing sets
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Assuming the sentiment features are not yet extracted and included in the CSV
# Apply Sentiment Analysis Function to extract features
def extract_sentiment_features(text):
    analysis = TextBlob(text)
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity

# Assuming 'edit_text' column exists in the loaded sets for feature extraction
if 'edit_text' in train_data.columns and 'edit_text' in test_data.columns:
    train_data['polarity'], train_data['subjectivity'] = zip(*train_data['edit_text'].apply(extract_sentiment_features))
    test_data['polarity'], test_data['subjectivity'] = zip(*test_data['edit_text'].apply(extract_sentiment_features))

# Preparing the feature set and labels from the loaded data
X_train = train_data[['polarity', 'subjectivity']]
y_train = train_data['is_sockpuppet']
X_test = test_data[['polarity', 'subjectivity']]
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.57      0.42      0.48      5382
           1       0.54      0.68      0.60      5382

    accuracy                           0.55     10764
   macro avg       0.55      0.55      0.54     10764
weighted avg       0.55      0.55      0.54     10764

Accuracy: 0.5508175399479748


## Integrated Semantic and Sentiment Analysis for Sockpuppet Detection in Wikipedia

This section combines semantic and sentiment analysis to enhance the accuracy of sockpuppet detection in Wikipedia. By integrating both analytical dimensions, the methodology seeks to provide a comprehensive view of the textual data, tapping into both the explicit meaning of the text (semantics) and the underlying emotions or attitudes (sentiment) to more effectively identify sockpuppet accounts.


In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time  # Import time module

# Load the pre-split training and testing datasets
train_data = pd.read_csv('wikipedia_sockpuppet_dataset_TRAIN.csv')
test_data = pd.read_csv('wikipedia_sockpuppet_dataset_TEST.csv')

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(pd.concat([train_data['edit_text'], test_data['edit_text']]))  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

Training completed in 83.315 seconds
              precision    recall  f1-score   support

           0       0.78      0.65      0.71      5382
           1       0.70      0.81      0.75      5382

    accuracy                           0.73     10764
   macro avg       0.74      0.73      0.73     10764
weighted avg       0.74      0.73      0.73     10764

Accuracy: 0.7312337421033073


## Raw Data

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('wikipedia_sockpuppet_dataset_UNCLEAN.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")

Training completed in 141.876 seconds
              precision    recall  f1-score   support

         0.0       0.82      0.97      0.89     17996
         1.0       0.84      0.39      0.53      6175

    accuracy                           0.82     24171
   macro avg       0.83      0.68      0.71     24171
weighted avg       0.83      0.82      0.80     24171

Accuracy: 0.8248727814322949
False Positive Rate (FPR): 0.026


## Lowercase

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('wikipedia_sockpuppet_dataset_UNCLEAN_Lowercase.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")

Training completed in 145.060 seconds
              precision    recall  f1-score   support

         0.0       0.82      0.97      0.89     17996
         1.0       0.83      0.39      0.53      6175

    accuracy                           0.82     24171
   macro avg       0.83      0.68      0.71     24171
weighted avg       0.82      0.82      0.80     24171

Accuracy: 0.8236316246741964
False Positive Rate (FPR): 0.028


## Symbols

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('wikipedia_sockpuppet_dataset_UNCLEAN_Symbols.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")

Training completed in 167.503 seconds
              precision    recall  f1-score   support

         0.0       0.81      0.97      0.88     17910
         1.0       0.80      0.33      0.47      6245

    accuracy                           0.81     24155
   macro avg       0.80      0.65      0.68     24155
weighted avg       0.81      0.81      0.78     24155

Accuracy: 0.8065824880977024
False Positive Rate (FPR): 0.029


## URL

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('wikipedia_sockpuppet_dataset_UNCLEAN_URL.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")

Training completed in 139.776 seconds
              precision    recall  f1-score   support

         0.0       0.76      0.97      0.85     12596
         1.0       0.79      0.30      0.44      5425

    accuracy                           0.77     18021
   macro avg       0.78      0.63      0.64     18021
weighted avg       0.77      0.77      0.73     18021

Accuracy: 0.7657732645247212
False Positive Rate (FPR): 0.034


## Balance

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('randomized_merged.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")

Training completed in 77.817 seconds
              precision    recall  f1-score   support

           0       0.76      0.64      0.70      5284
           1       0.70      0.80      0.75      5480

    accuracy                           0.72     10764
   macro avg       0.73      0.72      0.72     10764
weighted avg       0.73      0.72      0.72     10764

Accuracy: 0.7237086584912672
False Positive Rate (FPR): 0.357


## 90%

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import time  # Import time module

# Load the single dataset with specified encoding
data = pd.read_csv('90%.csv', encoding='latin1')

# Fill NaN values in the 'edit_text' column with a placeholder string
data['edit_text'] = data['edit_text'].fillna('')

# Fill NaN values in the target variable with the most frequent value
data['is_sockpuppet'] = data['is_sockpuppet'].fillna(data['is_sockpuppet'].mode()[0])

# Split the data into training and testing sets (80% training, 20% testing)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Feature extraction with TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
tfidf.fit(data['edit_text'])  # Fit on all available text
X_train_tfidf = tfidf.transform(train_data['edit_text']).toarray()
X_test_tfidf = tfidf.transform(test_data['edit_text']).toarray()

# Sentiment Analysis
train_data['polarity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
train_data['subjectivity'] = train_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
test_data['polarity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
test_data['subjectivity'] = test_data['edit_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

# Combining features
X_train = np.hstack((X_train_tfidf, train_data[['polarity', 'subjectivity']].values))
X_test = np.hstack((X_test_tfidf, test_data[['polarity', 'subjectivity']].values))

# Extracting labels
y_train = train_data['is_sockpuppet']
y_test = test_data['is_sockpuppet']

# Model Training using Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Start timing
start_time = time.time()

# Fit the model
model.fit(X_train, y_train)

# End timing
end_time = time.time()

# Calculate and print the training time
training_time = end_time - start_time
print(f"Training completed in {training_time:.3f} seconds")

# Predictions and Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# False positive rate (FPR) calculation
FP = cm[0, 1]  # False Positives
TN = cm[0, 0]  # True Negatives
FPR = FP / (FP + TN)

print(f"False Positive Rate (FPR): {FPR:.3f}")